# Get Features From Combine Tracking Data

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

This notebook serves as an intial broad overview of the relationship between Combine metrics (acceleration, deceleration, and cutting metrics that we've created) and in-game separation while running routes. 

We look at seperation at pass, max separation, separation created from start-of-play to throw, etc. 

We want to get a general sense for if any features are predictive of success at creating separation during man coverage siutations.

**<span style="color: gold;">FINDINGS</span>**

No meaningful findings here. 

This is likely because we are looking to broadly. Moving forward, we will get more specific and try to find signal in more specific situations.



<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

In [ ]:
# =========================================
# WR Players Data
# =========================================

wr_players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
)

wr_players_df = (
    wr_players_df
    .filter(pl.col("nfl_position") == "WR")
    .select("nfl_id", "nfl_position")
)

wr_players_df

In [ ]:
# =========================================
# Get Data For Targeted Passes
# =========================================

player_play_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_play.csv"), 
        schema_overrides = schema_map["combine_tracking"]
    )
)

player_play_df = (
    player_play_df
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "target", 
        "separation_at_pass_forward", 
        "team_coverage_man_zone",
        "yardline_number"
    )
    .filter(
        pl.col("target")
    )
    .filter(
        pl.col("team_coverage_man_zone") == "MAN_COVERAGE"
    )
    .filter(
        pl.col("yardline_number") >= 10
    )
)

player_play_df = (
    player_play_df
    .join(
        wr_players_df, 
        on = "nfl_id", 
        how = "inner"
    )
)

print("Player-Play Data:")
print(f"    - Number of Players In This Dataset: {player_play_df.select(pl.col("nfl_id")).n_unique()}")
player_play_df

In [ ]:
avg_separation_df = (
    player_play_df
    .group_by("nfl_id")
    .agg(avg_separation_at_pass = pl.mean("separation_at_pass_forward"))
)

avg_separation_df

In [ ]:
# =========================================
# WR Separation Stats Among 2023 Rookies
# =========================================

separation_df = pl.read_csv(str(SEPARATION_DATA_DIR / "02_2023_Rookie_Player_Play_Level_Separation_Data.csv"))

# -----------------------------------------
# Only Keep Plays Where The WR Is Targeted
# -----------------------------------------

separation_df = (
    separation_df
    .join(
        player_play_df, 
        on = ["game_id", "play_id", "nfl_id"], 
        how = "inner"
    )
)

# -----------------------------------------
# Only Keep Players With >= 10 Player-Plays
# -----------------------------------------

# players_with_good_exposure = (
#     separation_df
#     .group_by("nfl_id")
#     .agg(num_player_plays = pl.len())
#     .filter(pl.col("num_player_plays") >= 10)
#     .select("nfl_id")
#     .unique()
#     .to_series()
#     .to_list()
# )

# separation_df = (
#     separation_df
#     .filter(pl.col("nfl_id").is_in(players_with_good_exposure))
# )

print("Separation Data:")
print(f"    - Dimensions: {separation_df.shape}")
print(f"    - Number of Rookies: {separation_df.select("nfl_id").n_unique()}")
separation_df

In [ ]:
separation_avg_df = (
    separation_df
    .group_by("nfl_id")
    .agg(
        separation_avg = pl.mean("avg_separation"),
        separation_at_throw = pl.mean("separation_at_throw"),
        tight_coverage_pct = pl.mean("tight_coverage_pct"),
        separation_created_after_closest = pl.mean("separation_created_after_closest"),
        max_separation = pl.mean("max_separation"),
        min_separation = pl.mean("min_separation"),
        separation_at_start = pl.mean("separation_at_start"),
        separation_gain_start_to_throw = pl.mean("separation_gain_start_to_throw"),
        separation_sd = pl.mean("separation_sd"),
    )
)

separation_avg_df

In [ ]:
# =======================================
# Combine Features Data
# =======================================

combine_features_df = pl.read_csv(str(SEPARATION_DATA_DIR / "03_WR_Combine_Features_Data.csv"))

print("Combine Features:")
print(f"    - Dimensions: {combine_features_df.shape}")
print(f"    - Number of Rookies: {combine_features_df.select("nfl_id").n_unique()}")
combine_features_df

In [ ]:
# =====================================
# Join Separation and Features Data
# =====================================

analysis_df = (
    separation_df
    .join(
        combine_features_df, 
        on = "nfl_id", 
        how = "inner"
    )
)

print("Analysis Data:")
print(f"    - Dimensions: {analysis_df.shape}")
print(f"    - Number of Rookies: {analysis_df.select("nfl_id").n_unique()}")
analysis_df

In [ ]:
# =====================================
# Join Separation and Features Data
# =====================================

analysis_avg_df = (
    avg_separation_df
    .join(
        combine_features_df, 
        on = "nfl_id", 
        how = "inner"
    )
)

print("Analysis Data:")
print(f"    - Dimensions: {analysis_avg_df.shape}")
print(f"    - Number of Rookies: {analysis_avg_df.select("nfl_id").n_unique()}")
analysis_avg_df

<br><br/>
## Separation At Throw For All WRs (Not Just 2023 Rookies)
***

In [ ]:
fig = px.scatter(
    data_frame = analysis_avg_df, 
    x = "decel_score", 
    y = "avg_separation_at_pass", 
    trendline = "ols"
)

fig.show()

In [ ]:
import statsmodels.api as sm


predictor = "best_decel_to_velocity_ratio" 
outcome = "avg_separation_at_pass"

regression_df = analysis_avg_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
predictors = [
    "accel_score", 
    "decel_score", 
    "cut_score"
]

outcome = "avg_separation_at_pass"

regression_df = analysis_avg_df

# Keep rows where all predictors and the outcome are non-null
regression_df = regression_df.filter(
    pl.all_horizontal(
        [
            pl.col(col).is_not_null()
            for col in predictors + [outcome]
        ]
    )
)

# Convert to pandas
pd_reg = regression_df.to_pandas()

# Define predictors and outcome
X = pd_reg[predictors]
y = pd_reg[outcome]

# Add intercept
X = sm.add_constant(X)

# Fit multiple linear regression
model = sm.OLS(y, X).fit()

print(model.summary())